In [4]:
import json
import optuna
import optuna.visualization as vis
import pandas as pd
import tuning as tn

df = tn.load_tuning_data()

In [8]:
feats_full_h1 = tn.load_feature_set("full", df)
study_h1_full = tn.run_study(
    model_name="lightgbm", horizon=1, df=df, 
    features=feats_full_h1, feature_set="full", task="reg", owner="abdullah_alaa"
)

feats_pruned_h1 = tn.load_feature_set("pruned", df)
study_h1_pruned = tn.run_study(
    model_name="lightgbm", horizon=1, df=df, 
    features=feats_pruned_h1, feature_set="pruned", task="reg", owner="abdullah_alaa"
)

[I 2026-10-25 20:16:13,323] Using an existing study with `study_name='lgbm-reg-h1-full'` instead of creating a new one.


[I 2026-10-25 20:16:38,050] Trial 0 finished with value: 15.813274635058509 and parameters: {'num_leaves': 22, 'learning_rate': 0.17254716573280354, 'n_estimators': 600, 'min_child_samples': 29, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.49359671220172163, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 15.813274635058509.
[I 2026-10-25 20:16:41,034] Trial 1 finished with value: 15.227557050324569 and parameters: {'num_leaves': 41, 'learning_rate': 0.08341106432362087, 'n_estimators': 100, 'min_child_samples': 92, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.5274034664069657, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 15.227557050324569.
[I 2026-10-25 20:16:48,789] Trial 2 finished with value: 15.094552829241891 and parameters: {'num_leaves': 18, 'learning_rate': 0.048164145309070844, 'n_estimators': 400, 'min_child_samples': 11, 'subsample': 0.805926447

In [9]:
# Horizon 2 (48h) Tuning

feats_full_h2 = tn.load_feature_set("full", df)
study_h2_full = tn.run_study(
    model_name="lightgbm", horizon=2, df=df,
    features=feats_full_h2, feature_set="full", task="reg", owner="abdullah_alaa"
)

feats_pruned_h2 = tn.load_feature_set("pruned", df)
study_h2_pruned = tn.run_study(
    model_name="lightgbm", horizon=2, df=df,
    features=feats_pruned_h2, feature_set="pruned", task="reg", owner="abdullah_alaa"
)

# Horizon 3 (72h) Tuning

feats_full_h3 = tn.load_feature_set("full", df)
study_h3_full = tn.run_study(
    model_name="lightgbm", horizon=3, df=df,
    features=feats_full_h3, feature_set="full", task="reg", owner="abdullah_alaa"
)

feats_pruned_h3 = tn.load_feature_set("pruned", df)
study_h3_pruned = tn.run_study(
    model_name="lightgbm", horizon=3, df=df,
    features=feats_pruned_h3, feature_set="pruned", task="reg", owner="abdullah_alaa"
)

[I 2026-10-25 20:34:35,078] A new study created in RDB with name: lgbm-reg-h2-full
[I 2026-10-25 20:34:48,751] Trial 0 finished with value: 18.522296728660745 and parameters: {'num_leaves': 22, 'learning_rate': 0.17254716573280354, 'n_estimators': 600, 'min_child_samples': 29, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.49359671220172163, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 18.522296728660745.
[I 2026-10-25 20:34:53,151] Trial 1 finished with value: 17.24337021290376 and parameters: {'num_leaves': 41, 'learning_rate': 0.08341106432362087, 'n_estimators': 100, 'min_child_samples': 92, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.5274034664069657, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 17.24337021290376.
[I 2026-10-25 20:35:01,631] Trial 2 finished with value: 17.49185948897854 and parameters: {'num_leaves': 18, 'learning_rate': 0.0481641453

In [11]:
best_studies = {
    "t+1": study_h1_pruned if study_h1_pruned.best_value < study_h1_full.best_value else study_h1_full,
    "t+2": study_h2_pruned if study_h2_pruned.best_value < study_h2_full.best_value else study_h2_full,
    "t+3": study_h3_pruned if study_h3_pruned.best_value < study_h3_full.best_value else study_h3_full,
}

for h, study in best_studies.items():
    print(f"Optuna Visualizations for {h} (Best Study: {study.study_name})")
    
    fig_hist = vis.plot_optimization_history(study)
    fig_hist.show()
    
    fig_imp = vis.plot_param_importances(study)
    fig_imp.show()

Optuna Visualizations for t+1 (Best Study: lgbm-reg-h1-full)


Optuna Visualizations for t+2 (Best Study: lgbm-reg-h2-pruned)


Optuna Visualizations for t+3 (Best Study: lgbm-reg-h3-pruned)


In [19]:
for h, study in best_studies.items():
    fig_hist = vis.plot_optimization_history(study)
    fig_hist.write_html(f"optuna_history_{h}.html")
    
    fig_imp = vis.plot_param_importances(study)
    fig_imp.write_html(f"optuna_importance_{h}.html")

print("All plots saved as HTMLs successfully!")

All plots saved as HTMLs successfully!


In [22]:
studies = {
    ("t+1", "full"): study_h1_full,
    ("t+1", "pruned"): study_h1_pruned,
    ("t+2", "full"): study_h2_full,
    ("t+2", "pruned"): study_h2_pruned,
    ("t+3", "full"): study_h3_full,
    ("t+3", "pruned"): study_h3_pruned,
}

records = []
for (h, fset), s in studies.items():
    records.append({
        "horizon": h,
        "feature_set": fset,
        "tuned_cv_rmse": s.best_value,
        "best_params": s.best_params,
    })

summary_df = pd.DataFrame(records)

baseline_df = pd.read_csv("artifacts/results/task1_default_baseline_cv.csv")
task1_lgb = baseline_df[
    (baseline_df["model"] == "lightgbm") & (baseline_df["task"] == "reg")
][["horizon", "feature_set", "rmse_mean", "rmse_std"]].copy()

task1_lgb["horizon"] = task1_lgb["horizon"].apply(lambda x: f"t+{x}" if not str(x).startswith("t+") else str(x))

task1_lgb.rename(
    columns={
        "rmse_mean": "default_rmse_mean",
        "rmse_std": "default_rmse_std",
    },
    inplace=True,
)

comparison_df = pd.merge(summary_df, task1_lgb, on=["horizon", "feature_set"])
comparison_df["improvement_%"] = (
    (comparison_df["default_rmse_mean"] - comparison_df["tuned_cv_rmse"])
    / comparison_df["default_rmse_mean"]
    * 100
)

print("=== Task 2: Tuned LightGBM vs Week 4 Default Baseline ===")
print(
    comparison_df[[
        "horizon",
        "feature_set",
        "default_rmse_mean",
        "tuned_cv_rmse",
        "improvement_%",
    ]]
)

best_configs = {}
for h in ["t+1", "t+2", "t+3"]:
    sub = comparison_df[comparison_df["horizon"] == h]
    best_row = sub.sort_values(by="tuned_cv_rmse").iloc[0]
    best_configs[h] = {
        "chosen_feature_set": best_row["feature_set"],
        "default_cv_rmse": float(best_row["default_rmse_mean"]),
        "tuned_cv_rmse": float(best_row["tuned_cv_rmse"]),
        "improvement_%": float(best_row["improvement_%"]),
        "best_params": best_row["best_params"],
    }

with open("task2_best_lightgbm_params.json", "w") as f:
    json.dump(best_configs, f, indent=4)

print("\nSaved: task2_best_lightgbm_params.json")

=== Task 2: Tuned LightGBM vs Week 4 Default Baseline ===
  horizon feature_set  default_rmse_mean  tuned_cv_rmse  improvement_%
0     t+1        full          15.400749      14.925884       3.083388
1     t+1      pruned          15.400749      14.930028       3.056482
2     t+2        full          17.652315      16.920075       4.148125
3     t+2      pruned          17.597817      16.909047       3.913957
4     t+3        full          18.068282      17.537295       2.938781
5     t+3      pruned          18.132409      17.533235       3.304433

Saved: task2_best_lightgbm_params.json
